# External dataset validation: real ADS-B flight vs. classical / IMM / adaptive filters

Self-contained Colab notebook for the "another dataset" generalization test described in
Section VI of the paper. Bundles every dependency from the project's `radar_kf_project/`
codebase inline, so this runs standalone without uploading the rest of the repo -- you only
need to upload one file: your trained model checkpoint (`adaptive_kf_standardized_1400.pt`,
from `radar_kf_project/outputs/` in the code package).

**What this does:** pulls a real aircraft's historical trajectory from OpenSky, treats it as
ground truth, runs it through this project's own noise/detection simulator, and evaluates the
classical, IMM, and adaptive filters against it -- the same evaluation used everywhere else in
the paper, just with a real flight path instead of a synthetic one.

Run the cells top to bottom.

## 1. Install dependencies

Colab already has `numpy`, `torch`, `scipy`, and `matplotlib` preinstalled. `requests` usually is too, but this ensures it.

In [ ]:
!pip install -q requests

## 2. OpenSky credentials

From your OpenSky account's **API Client** page (https://opensky-network.org/my-opensky/account). The secret is entered via a hidden prompt so it isn't stored in the notebook file.

In [ ]:
import os
from getpass import getpass

os.environ["OPENSKY_CLIENT_ID"] = input("OpenSky client_id: ").strip()
os.environ["OPENSKY_CLIENT_SECRET"] = getpass("OpenSky client_secret (hidden): ").strip()
print("Credentials set for this session.")

## 3. Upload your trained model checkpoint

Upload `adaptive_kf_standardized_1400.pt` from `radar_kf_project/outputs/` in the code package (the standardized-protocol headline model Table I is built from).

In [ ]:
from google.colab import files

uploaded = files.upload()
CHECKPOINT_PATH = list(uploaded.keys())[0]
print(f"Using checkpoint: {CHECKPOINT_PATH}")

## 4. Project modules (bundled inline)

Everything below is copied verbatim from the tested project codebase (`config.py` through
`adaptive_kf.py`) -- nothing rewritten, only reorganized into cells so this notebook needs no
other files. If you already have the full `radar_kf_project/` folder in this Colab environment
instead, skip to Section 5 and `import` these modules normally rather than running this section.

### `config.py` -- simulation parameters (single source of truth, unchanged from the project)

In [ ]:
import numpy as np

# ---------------- Scenario timing ----------------
T = 1.0                 # scan interval (s)
N = 100                  # number of scans

# ---------------- True target motion (constant velocity) ----------------
VX_TRUE = 150.0          # m/s
VY_TRUE = 70.0           # m/s
X0, Y0 = 60000.0, 25000.0  # initial position (m) -- radar assumed at origin.

# ---------------- Measurement noise (polar domain) ----------------
SIGMA_R = 50.0                        # range noise std dev (m)
SIGMA_THETA_DEG = 0.5                 # azimuth noise std dev (deg)
SIGMA_THETA = np.deg2rad(SIGMA_THETA_DEG)

# ---------------- Detection model ----------------
PD = 0.92                # probability of detection (Bernoulli)
LAMBDA_FA = 1.5           # mean false alarms per scan (Poisson rate)
FA_BOX_MARGIN = 3000.0   # meters beyond trajectory bounding box, for false alarm placement

# ---------------- Kalman filter tuning ----------------
Q_SCALE = 0.5             # process noise intensity (white-noise-acceleration model)
P0_POS_STD = 500.0        # initial position uncertainty (m) std dev
P0_VEL_STD = 200.0        # initial velocity uncertainty (m/s) std dev

# ---------------- Monte Carlo ----------------
MC_RUNS = 300             # number of independent Monte Carlo trials

# ---------------- Reproducibility ----------------
BASE_SEED = 42

# Wrap constants in a lightweight namespace so the rest of this notebook
# can use `cfg.X` exactly like the project's `import config as cfg` does.
class _Config:
    pass

cfg = _Config()
for _name, _val in list(globals().items()):
    if _name.isupper():
        setattr(cfg, _name, _val)
print("config loaded:", [n for n in dir(cfg) if n.isupper()])

### `measurement_model.py` -- polar-domain measurement noise + converted covariance

In [ ]:
def generate_noisy_measurements(x_true, y_true, rng):
    """Add Gaussian noise in polar domain, return noisy (x, y) plus the
    underlying noisy (r, theta) and true (r, theta) for diagnostics."""
    r_true = np.sqrt(x_true**2 + y_true**2)
    theta_true = np.arctan2(y_true, x_true)

    r_meas = r_true + rng.normal(0, cfg.SIGMA_R, cfg.N)
    theta_meas = theta_true + rng.normal(0, cfg.SIGMA_THETA, cfg.N)

    x_meas = r_meas * np.cos(theta_meas)
    y_meas = r_meas * np.sin(theta_meas)
    return x_meas, y_meas, r_meas, theta_meas, r_true, theta_true


def measurement_covariance(r, theta):
    """Converted-measurement covariance R via first-order (Jacobian)
    linearization of the polar -> Cartesian transform, evaluated at the
    (noisy) measured range/azimuth."""
    J = np.array([[np.cos(theta), -r * np.sin(theta)],
                  [np.sin(theta),  r * np.cos(theta)]])
    polar_cov = np.diag([cfg.SIGMA_R**2, cfg.SIGMA_THETA**2])
    return J @ polar_cov @ J.T

### `detection_model.py` -- Bernoulli missed detections

In [ ]:
def apply_missed_detections(x_meas, y_meas, rng, pd=None):
    """Returns (x_target_meas, y_target_meas, detected_mask). Missed
    scans are set to NaN in the returned arrays."""
    pd = cfg.PD if pd is None else pd
    detected = rng.random(cfg.N) < pd
    x_target_meas = np.where(detected, x_meas, np.nan)
    y_target_meas = np.where(detected, y_meas, np.nan)
    return x_target_meas, y_target_meas, detected

### `kalman_filter.py` -- classical constant-velocity Kalman filter

In [ ]:
class ConstantVelocityKF:
    def __init__(self):
        self.F = np.array([[1, 0, cfg.T, 0],
                            [0, 1, 0, cfg.T],
                            [0, 0, 1, 0],
                            [0, 0, 0, 1]])
        self.H = np.array([[1, 0, 0, 0],
                            [0, 1, 0, 0]])
        q = cfg.Q_SCALE
        T = cfg.T
        self.Q = q * np.array([
            [T**3/3, 0,      T**2/2, 0],
            [0,      T**3/3, 0,      T**2/2],
            [T**2/2, 0,      T,      0],
            [0,      T**2/2, 0,      T],
        ])
        self.x_est = None
        self.P_est = None

    def initialize(self, x0, y0):
        self.x_est = np.array([x0, y0, 0.0, 0.0])
        self.P_est = np.diag([cfg.P0_POS_STD**2, cfg.P0_POS_STD**2,
                               cfg.P0_VEL_STD**2, cfg.P0_VEL_STD**2])

    def step(self, z=None, R=None):
        x_pred = self.F @ self.x_est
        P_pred = self.F @ self.P_est @ self.F.T + self.Q

        if z is not None:
            innovation = z - self.H @ x_pred
            S = self.H @ P_pred @ self.H.T + R
            K = P_pred @ self.H.T @ np.linalg.inv(S)
            self.x_est = x_pred + K @ innovation
            self.P_est = (np.eye(4) - K @ self.H) @ P_pred
        else:
            self.x_est = x_pred
            self.P_est = P_pred
        return self.x_est.copy()


def run_filter_with_covariance(x_target_meas, y_target_meas, detected,
                                 r_meas, theta_meas, measurement_covariance_fn,
                                 q_scale=None):
    """Returns (x_history, P_history) where P_history has shape (N, 4, 4)."""
    kf = ConstantVelocityKF()
    if q_scale is not None:
        q = q_scale
        T = cfg.T
        kf.Q = q * np.array([
            [T**3/3, 0,      T**2/2, 0],
            [0,      T**3/3, 0,      T**2/2],
            [T**2/2, 0,      T,      0],
            [0,      T**2/2, 0,      T],
        ])
    first_idx = int(np.argmax(detected))
    kf.initialize(x_target_meas[first_idx], y_target_meas[first_idx])

    x_history = np.zeros((cfg.N, 4))
    P_history = np.zeros((cfg.N, 4, 4))
    for i in range(cfg.N):
        if detected[i]:
            z = np.array([x_target_meas[i], y_target_meas[i]])
            R = measurement_covariance_fn(r_meas[i], theta_meas[i])
            x_history[i] = kf.step(z, R)
        else:
            x_history[i] = kf.step(None, None)
        P_history[i] = kf.P_est
    return x_history, P_history

### `imm_filter.py` -- classical two-model IMM baseline (Bar-Shalom et al.)

In [ ]:
T = cfg.T
F = np.array([[1, 0, T, 0], [0, 1, 0, T], [0, 0, 1, 0], [0, 0, 0, 1]])
H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]])
Q_UNIT = np.array([
    [T**3/3, 0,      T**2/2, 0],
    [0,      T**3/3, 0,      T**2/2],
    [T**2/2, 0,      T,      0],
    [0,      T**2/2, 0,      T],
])

DEFAULT_TRANSITION = np.array([[0.95, 0.05],
                                [0.05, 0.95]])  # standard textbook default


def gaussian_likelihood(innovation, S):
    d = innovation.shape[0]
    det_S = np.linalg.det(S)
    if det_S <= 0:
        det_S = 1e-12
    norm_const = 1.0 / np.sqrt((2 * np.pi) ** d * det_S)
    exponent = -0.5 * innovation @ np.linalg.solve(S, innovation)
    return norm_const * np.exp(exponent)


class IMMFilter:
    def __init__(self, q_low=None, q_high=None, transition=None):
        self.q_low = cfg.Q_SCALE if q_low is None else q_low
        self.q_high = 500.0 if q_high is None else q_high  # selected via select_q_high.py grid search
        self.Pi = DEFAULT_TRANSITION if transition is None else transition
        self.Q = [self.q_low * Q_UNIT, self.q_high * Q_UNIT]
        self.n_models = 2

    def initialize(self, x0, y0):
        x_init = np.array([x0, y0, 0.0, 0.0])
        P_init = np.diag([cfg.P0_POS_STD**2, cfg.P0_POS_STD**2,
                           cfg.P0_VEL_STD**2, cfg.P0_VEL_STD**2])
        self.x = [x_init.copy(), x_init.copy()]
        self.P = [P_init.copy(), P_init.copy()]
        self.mu = np.array([0.5, 0.5])

    def step(self, z=None, R=None):
        c = self.Pi.T @ self.mu
        mix_w = np.zeros((self.n_models, self.n_models))
        for j in range(self.n_models):
            for i in range(self.n_models):
                mix_w[i, j] = self.Pi[i, j] * self.mu[i] / c[j]

        x0j, P0j = [], []
        for j in range(self.n_models):
            xm = sum(mix_w[i, j] * self.x[i] for i in range(self.n_models))
            Pm = np.zeros((4, 4))
            for i in range(self.n_models):
                diff = (self.x[i] - xm).reshape(-1, 1)
                Pm += mix_w[i, j] * (self.P[i] + diff @ diff.T)
            x0j.append(xm)
            P0j.append(Pm)

        x_new, P_new, likelihoods = [], [], []
        for j in range(self.n_models):
            x_pred = F @ x0j[j]
            P_pred = F @ P0j[j] @ F.T + self.Q[j]
            if z is not None:
                innovation = z - H @ x_pred
                S = H @ P_pred @ H.T + R
                K = P_pred @ H.T @ np.linalg.inv(S)
                xj = x_pred + K @ innovation
                Pj = (np.eye(4) - K @ H) @ P_pred
                lam = gaussian_likelihood(innovation, S)
            else:
                xj, Pj, lam = x_pred, P_pred, 1.0
            x_new.append(xj); P_new.append(Pj); likelihoods.append(lam)

        num = np.array(likelihoods) * c
        self.mu = num / num.sum()

        x_comb = sum(self.mu[j] * x_new[j] for j in range(self.n_models))
        P_comb = np.zeros((4, 4))
        for j in range(self.n_models):
            diff = (x_new[j] - x_comb).reshape(-1, 1)
            P_comb += self.mu[j] * (P_new[j] + diff @ diff.T)

        self.x, self.P = x_new, P_new
        return x_comb, P_comb, self.mu.copy()


def run_imm(x_target_meas, y_target_meas, detected, r_meas, theta_meas,
            measurement_covariance_fn, q_low=None, q_high=None, transition=None):
    imm = IMMFilter(q_low=q_low, q_high=q_high, transition=transition)
    first_idx = int(np.argmax(detected))
    imm.initialize(x_target_meas[first_idx], y_target_meas[first_idx])

    x_history = np.zeros((cfg.N, 4))
    P_history = np.zeros((cfg.N, 4, 4))
    mu_history = np.zeros((cfg.N, 2))
    for i in range(cfg.N):
        if detected[i]:
            z = np.array([x_target_meas[i], y_target_meas[i]])
            R = measurement_covariance_fn(r_meas[i], theta_meas[i])
            x_history[i], P_history[i], mu_history[i] = imm.step(z, R)
        else:
            x_history[i], P_history[i], mu_history[i] = imm.step(None, None)
    return x_history, P_history, mu_history

### `metrics.py` -- NEES consistency metric

In [ ]:
def compute_nees(x_true_seq, x_est_seq, P_seq, state_dims=None):
    """Per-scan NEES for one run. Returns an (N,) array."""
    Nn = x_true_seq.shape[0]
    if state_dims is None:
        state_dims = list(range(x_true_seq.shape[1]))
    nees = np.zeros(Nn)
    for k in range(Nn):
        err = (x_true_seq[k, state_dims] - x_est_seq[k, state_dims])
        P_sub = P_seq[k][np.ix_(state_dims, state_dims)]
        nees[k] = err @ np.linalg.solve(P_sub, err)
    return nees

### `adaptive_kf.py` -- the learned adaptive-Q filter (GRU-based)

In [ ]:
import torch
import torch.nn as nn

_T = cfg.T
F_np = np.array([[1, 0, _T, 0], [0, 1, 0, _T], [0, 0, 1, 0], [0, 0, 0, 1]])
H_np = np.array([[1, 0, 0, 0], [0, 1, 0, 0]])
Q_UNIT_np = np.array([
    [_T**3/3, 0,       _T**2/2, 0],
    [0,       _T**3/3, 0,       _T**2/2],
    [_T**2/2, 0,       _T,      0],
    [0,       _T**2/2, 0,       _T],
])


class AdaptiveKF(nn.Module):
    def __init__(self, hidden_size=16, base_q=cfg.Q_SCALE, normalize_innovation=False):
        super().__init__()
        self.hidden_size = hidden_size
        self.base_q = base_q
        self.normalize_innovation = normalize_innovation
        self.gru = nn.GRUCell(input_size=3, hidden_size=hidden_size)
        self.head = nn.Linear(hidden_size, 1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

        self.F = torch.tensor(F_np, dtype=torch.float32)
        self.H = torch.tensor(H_np, dtype=torch.float32)
        self.Q_unit = torch.tensor(Q_UNIT_np, dtype=torch.float32)
        self.I4 = torch.eye(4)

    def forward(self, traj):
        z = torch.tensor(traj["z"], dtype=torch.float32)
        R = torch.tensor(traj["R"], dtype=torch.float32)
        detected = traj["detected"]
        Nn = z.shape[0]

        first_idx = int(np.argmax(detected))
        x_est = torch.tensor(
            [traj["z"][first_idx, 0], traj["z"][first_idx, 1], 0.0, 0.0],
            dtype=torch.float32,
        )
        P_est = torch.diag(torch.tensor(
            [cfg.P0_POS_STD**2, cfg.P0_POS_STD**2, cfg.P0_VEL_STD**2, cfg.P0_VEL_STD**2],
            dtype=torch.float32,
        ))
        h = torch.zeros(self.hidden_size)

        x_history, q_history, P_history = [], [], []

        for k in range(Nn):
            raw = self.head(h).squeeze(0) if h.dim() > 1 else self.head(h)
            q_scale = self.base_q * torch.exp(raw.squeeze())
            q_history.append(q_scale)

            Q_k = q_scale * self.Q_unit
            x_pred = self.F @ x_est
            P_pred = self.F @ P_est @ self.F.T + Q_k

            if detected[k]:
                innovation = z[k] - self.H @ x_pred
                S = self.H @ P_pred @ self.H.T + R[k]
                K = P_pred @ self.H.T @ torch.linalg.inv(S)
                x_est = x_pred + K @ innovation
                P_est = (self.I4 - K @ self.H) @ P_pred
                if self.normalize_innovation:
                    innov_feature = innovation / torch.sqrt(torch.diagonal(S) + 1e-8)
                else:
                    innov_feature = innovation
                gru_input = torch.cat([innov_feature, torch.tensor([1.0])])
            else:
                x_est = x_pred
                P_est = P_pred
                gru_input = torch.zeros(3)

            x_history.append(x_est)
            P_history.append(P_est)
            h = self.gru(gru_input.unsqueeze(0), h.unsqueeze(0) if h.dim() == 1 else h).squeeze(0)

        return torch.stack(x_history), torch.stack(q_history), torch.stack(P_history)


def run_adaptive_filter_numpy(model, traj):
    model.eval()
    with torch.no_grad():
        x_hist, q_hist, P_hist = model(traj)
    return x_hist.numpy(), q_hist.numpy(), P_hist.numpy()


print("All project modules loaded.")

## 5. OpenSky fetch + real-trajectory evaluation

Everything below is copied verbatim from `external_dataset_opensky.py`.

In [ ]:
import time as time_module
import requests

TOKEN_URL = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"


def latlon_to_local_xy(lat, lon, lat0, lon0):
    """Simple equirectangular flat-Earth projection, adequate for the
    short ranges (tens of km) this project's radar model assumes.
    Returns (x, y) in meters, x = east, y = north, origin at (lat0, lon0)."""
    R_EARTH = 6371000.0
    lat_rad, lon_rad = np.radians(lat), np.radians(lon)
    lat0_rad, lon0_rad = np.radians(lat0), np.radians(lon0)
    x = R_EARTH * (lon_rad - lon0_rad) * np.cos(lat0_rad)
    y = R_EARTH * (lat_rad - lat0_rad)
    return x, y


class TokenManager:
    """Handles OAuth2 client-credentials auth for a standard OpenSky
    account. Tokens expire after 30 minutes and are refreshed automatically."""

    def __init__(self):
        self.client_id = os.environ.get("OPENSKY_CLIENT_ID")
        self.client_secret = os.environ.get("OPENSKY_CLIENT_SECRET")
        if not self.client_id or not self.client_secret:
            raise RuntimeError("Set OPENSKY_CLIENT_ID / OPENSKY_CLIENT_SECRET first (Section 2).")
        self.token = None
        self.expires_at = 0

    def headers(self):
        if self.token is None or time_module.time() > self.expires_at - 30:
            self._refresh()
        return {"Authorization": f"Bearer {self.token}"}

    def _refresh(self):
        r = requests.post(
            TOKEN_URL,
            data={
                "grant_type": "client_credentials",
                "client_id": self.client_id,
                "client_secret": self.client_secret,
            },
        )
        r.raise_for_status()
        data = r.json()
        self.token = data["access_token"]
        self.expires_at = time_module.time() + data.get("expires_in", 1800)


def fetch_opensky_trajectory(icao24, unix_time, tokens=None):
    """Fetches a real historical trajectory via GET /tracks and returns
    (x_true, y_true) of length config.N, in this project's coordinate
    convention (radar at origin, target at realistic range).

    icao24: lowercase hex transponder address, e.g. "4b1805"
    unix_time: any Unix timestamp (seconds) that falls within the
        flight you want -- /tracks returns the whole track containing
        that moment. Must be within the last 30 days per OpenSky's
        /tracks limitation."""
    if tokens is None:
        tokens = TokenManager()

    resp = requests.get(
        "https://opensky-network.org/api/tracks",
        params={"icao24": icao24, "time": unix_time},
        headers=tokens.headers(),
    )
    resp.raise_for_status()
    track = resp.json()

    path = track["path"]  # list of [time, lat, lon, baro_altitude, true_track, on_ground]
    if len(path) < 2:
        raise ValueError(
            f"Track for icao24={icao24} at time={unix_time} has too few waypoints "
            f"({len(path)}) to interpolate. Try a different timestamp or aircraft."
        )

    t = np.array([p[0] for p in path], dtype=float)
    lat = np.array([p[1] for p in path], dtype=float)
    lon = np.array([p[2] for p in path], dtype=float)

    valid = ~(np.isnan(lat) | np.isnan(lon))
    t, lat, lon = t[valid], lat[valid], lon[valid]
    order = np.argsort(t)
    t, lat, lon = t[order], lat[order], lon[order]

    lat0, lon0 = lat[0], lon[0]
    x, y = latlon_to_local_xy(lat, lon, lat0, lon0)
    x = x - x.min() + cfg.X0
    y = y - y.min() + cfg.Y0

    t_rel = t - t[0]
    t_grid = np.linspace(0, (cfg.N - 1) * cfg.T, cfg.N)
    if t_rel[-1] < t_grid[-1]:
        raise ValueError(
            f"Track only spans {t_rel[-1]:.0f}s, need at least "
            f"{(cfg.N - 1) * cfg.T:.0f}s for {cfg.N} scans at T={cfg.T}s. "
            f"Pick a longer flight or a different timestamp."
        )
    x_resampled = np.interp(t_grid, t_rel, x)
    y_resampled = np.interp(t_grid, t_rel, y)
    return x_resampled, y_resampled


def evaluate_on_real_trajectory(x_true, y_true, model, q_high=500, n_runs=150):
    """Same evaluation as evaluate_s_turn.py, but x_true/y_true come from
    a real flight instead of our synthetic truth generator. Noise, missed
    detections, and clutter are still drawn from OUR simulator, with a
    fresh random seed per Monte Carlo run -- only the underlying motion is real."""
    errs_c, errs_imm, errs_a = [], [], []
    nees_c, nees_imm, nees_a = [], [], []

    for run in range(n_runs):
        rng = np.random.default_rng(cfg.BASE_SEED + run)
        x_meas, y_meas, r_meas, theta_meas, r_true, theta_true = \
            generate_noisy_measurements(x_true, y_true, rng)
        x_target_meas, y_target_meas, detected = apply_missed_detections(x_meas, y_meas, rng)

        vx_true = np.gradient(x_true, cfg.T)
        vy_true = np.gradient(y_true, cfg.T)
        x_true_full = np.stack([x_true, y_true, vx_true, vy_true], axis=1)

        x_est_c, P_hist_c = run_filter_with_covariance(
            x_target_meas, y_target_meas, detected, r_meas, theta_meas, measurement_covariance)
        errs_c.append(np.sqrt((x_est_c[:, 0] - x_true) ** 2 + (x_est_c[:, 1] - y_true) ** 2))
        nees_c.append(compute_nees(x_true_full, x_est_c, P_hist_c, state_dims=[0, 1]))

        x_est_imm, P_hist_imm, _ = run_imm(
            x_target_meas, y_target_meas, detected, r_meas, theta_meas,
            measurement_covariance, q_high=q_high)
        errs_imm.append(np.sqrt((x_est_imm[:, 0] - x_true) ** 2 + (x_est_imm[:, 1] - y_true) ** 2))
        nees_imm.append(compute_nees(x_true_full, x_est_imm, P_hist_imm, state_dims=[0, 1]))

        R_seq = np.zeros((cfg.N, 2, 2))
        for k in range(cfg.N):
            if detected[k]:
                R_seq[k] = measurement_covariance(r_meas[k], theta_meas[k])
        z_seq = np.stack([np.where(detected, x_target_meas, 0.0),
                           np.where(detected, y_target_meas, 0.0)], axis=1)
        traj = {"x_true": x_true_full, "z": z_seq, "R": R_seq, "detected": detected}
        x_est_a, _, P_hist_a = run_adaptive_filter_numpy(model, traj)
        errs_a.append(np.sqrt((x_est_a[:, 0] - x_true) ** 2 + (x_est_a[:, 1] - y_true) ** 2))
        nees_a.append(compute_nees(x_true_full, x_est_a, P_hist_a, state_dims=[0, 1]))

    return (np.array(errs_c), np.array(errs_imm), np.array(errs_a),
            np.array(nees_c), np.array(nees_imm), np.array(nees_a))

print("Fetch and evaluation functions ready.")

## 6. Pick your flight and run

Find a flight with a clear turn near an airport via
[OpenSky's network explorer](https://opensky-network.org/network/explorer) or
[flightradar24.com](https://www.flightradar24.com), and note its **ICAO24 hex address** and
any **Unix timestamp** during that flight.

**Important:** `/tracks` only covers the last 30 days, and OpenSky processes historical data
in a nightly batch -- a flight from the last ~24h may not be available yet. The cell below
computes a *guaranteed-valid* timestamp (3 days ago) automatically; replace `ICAO24` with a
real aircraft you've confirmed was flying around that time, or adjust `DAYS_AGO` to match
whichever flight you actually found.

In [ ]:
import datetime

ICAO24 = "4b1805"   # <-- replace with your chosen aircraft's hex address
DAYS_AGO = 3          # <-- adjust to match when your chosen flight actually happened
UNIX_TIME = int((datetime.datetime.now(datetime.timezone.utc)
                  - datetime.timedelta(days=DAYS_AGO)).timestamp())
N_RUNS = 150           # Monte Carlo noise realizations (matches evaluate_s_turn.py)

print(f"Using UNIX_TIME={UNIX_TIME} ({DAYS_AGO} days ago) -- adjust DAYS_AGO if this doesn't match your chosen flight.")
print(f"Fetching real ADS-B track for icao24={ICAO24} covering time={UNIX_TIME}...")
x_true, y_true = fetch_opensky_trajectory(ICAO24, UNIX_TIME)
print(f"Got a {len(x_true)}-scan trajectory.")

model = AdaptiveKF(hidden_size=16)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location="cpu"))
model.eval()

print(f"Evaluating classical/IMM/adaptive ({N_RUNS} MC noise runs)...")
errs_c, errs_imm, errs_a, nees_c, nees_imm, nees_a = evaluate_on_real_trajectory(
    x_true, y_true, model, n_runs=N_RUNS)

print()
print(f"Classical: RMSE={np.sqrt(np.mean(errs_c**2)):.1f}m  NEES={nees_c.mean():.2f}")
print(f"IMM:       RMSE={np.sqrt(np.mean(errs_imm**2)):.1f}m  NEES={nees_imm.mean():.2f}")
print(f"Adaptive:  RMSE={np.sqrt(np.mean(errs_a**2)):.1f}m  NEES={nees_a.mean():.2f}")

## 7. Plot and save results

In [ ]:
import matplotlib.pyplot as plt
import pickle

k = np.arange(cfg.N)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k, errs_c.mean(axis=0), "r-", alpha=0.6,
        label=f"Classical (RMSE={np.sqrt(np.mean(errs_c**2)):.0f}m)")
ax.plot(k, errs_imm.mean(axis=0), color="purple", linewidth=1.8,
        label=f"IMM (RMSE={np.sqrt(np.mean(errs_imm**2)):.0f}m)")
ax.plot(k, errs_a.mean(axis=0), "b-", linewidth=1.8,
        label=f"Adaptive (RMSE={np.sqrt(np.mean(errs_a**2)):.0f}m)")
ax.set_xlabel("Scan number"); ax.set_ylabel("Mean position error (m)")
ax.set_title(f"Real ADS-B flight (icao24={ICAO24}) vs. simulator-only training")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("real_trajectory_comparison.png", dpi=120)
plt.show()

with open("real_trajectory_results.pkl", "wb") as f:
    pickle.dump({
        "icao24": ICAO24, "unix_time": UNIX_TIME, "x_true": x_true, "y_true": y_true,
        "classical_err": errs_c, "imm_err": errs_imm, "adaptive_err": errs_a,
        "classical_nees": nees_c, "imm_nees": nees_imm, "adaptive_nees": nees_a,
    }, f)
print("Saved real_trajectory_results.pkl and real_trajectory_comparison.png")

# Download both to your machine:
from google.colab import files as _files
_files.download("real_trajectory_results.pkl")
_files.download("real_trajectory_comparison.png")